# 📋 Parcial 2 — Tema A

**Materia:** Ciencia de Datos  
**Modalidad:** Individual · En clase

---

## Instrucciones

- Completar el campo `NRO_MATRICULA` con tu número de matrícula **antes de comenzar**.
- El dataset se genera a partir de tu matrícula — cada alumno trabaja con datos propios.
- Los resultados numéricos serán distintos entre alumnos. Se evalúa el razonamiento y la interpretación.
- **No modificar** las celdas marcadas como `# ── NO MODIFICAR ──`.

**- Recorda comentar el código**

---

## Contexto

Una empresa de tecnología registra información de sus empleados y quiere predecir quiénes tienen mayor riesgo de renunciar para actuar de forma preventiva.

## Variables del dataset

| Variable | Tipo | Descripción |
|----------|------|-------------|
| `legajo` | Numérico | Identificador único del empleado |
| `edad` | Numérico | Edad en años |
| `salario_mensual` | Numérico | Salario mensual en $ |
| `antiguedad` | Numérico | Años en la empresa |
| `horas_semanales` | Numérico | Promedio de horas trabajadas por semana |
| `satisfaccion` | Numérico | Score de satisfacción laboral (1 = muy bajo, 5 = muy alto) |
| `cant_proyectos` | Numérico | Cantidad de proyectos activos |
| `area` | Categórico | Área de la empresa |
| `cargo` | Categórico | Nivel jerárquico |
| `viaja_trabajo` | Categórico | Frecuencia de viajes de negocios |
| `renuncia` | **Target** | 1 = renunció, 0 = sigue en la empresa |

In [ ]:
# ══════════════════════════════════════════
#   COMPLETAR CON TU NÚMERO DE MATRÍCULA
# ══════════════════════════════════════════
NRO_MATRICULA = 0   # ← reemplazar con tu matrícula
# ══════════════════════════════════════════

In [ ]:
# ── NO MODIFICAR ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 110

def generar_dataset(n=1000, seed=42):
    rng = np.random.default_rng(seed)
    legajo         = rng.integers(10000, 99999, n)
    edad           = rng.integers(22, 60, n).astype(float)
    antiguedad     = np.clip(rng.normal(5, 4, n), 0, 30).round(1)
    salario        = rng.normal(150000, 60000, n).clip(40000, 400000).round(-2)
    horas          = rng.normal(45, 8, n).clip(30, 70).round(1)
    satisfaccion   = rng.integers(1, 6, n).astype(float)
    cant_proyectos = rng.integers(1, 9, n).astype(float)
    areas  = rng.choice(['Ventas','IT','RRHH','Finanzas','Operaciones'], n, p=[0.25,0.25,0.15,0.15,0.20])
    cargos = rng.choice(['Junior','Semi-Senior','Senior','Manager'], n, p=[0.30,0.30,0.25,0.15])
    viajes = rng.choice(['Nunca','Raramente','Frecuentemente'], n, p=[0.40,0.35,0.25])
    logit = (
        - 0.020 * (salario/1000 - 150)
        - 0.50  * satisfaccion
        + 0.06  * (horas - 45)
        + 0.80  * (viajes == 'Frecuentemente').astype(float)
        + 0.20  * (viajes == 'Raramente').astype(float)
        + 0.50  * (areas == 'Ventas').astype(float)
        + 0.20  * (areas == 'Operaciones').astype(float)
        - 0.50  * (cargos == 'Manager').astype(float)
        - 0.20  * (cargos == 'Senior').astype(float)
        - 0.02  * antiguedad
        + 0.5
        + rng.normal(0, 0.4, n)
    )
    prob     = 1 / (1 + np.exp(-logit))
    renuncia = rng.binomial(1, prob, n)
    return pd.DataFrame({
        'legajo': legajo, 'edad': edad, 'salario_mensual': salario,
        'antiguedad': antiguedad, 'horas_semanales': horas,
        'satisfaccion': satisfaccion, 'cant_proyectos': cant_proyectos,
        'area': areas, 'cargo': cargos, 'viaja_trabajo': viajes,
        'renuncia': renuncia
    })

df = generar_dataset(seed=NRO_MATRICULA)
print(f"Dataset generado con matrícula {NRO_MATRICULA}: {df.shape}")

---

## Punto 1 — Análisis estadístico descriptivo

**1a.** Realizar un análisis descriptivo de las variables numéricas del dataset.

**1b.** Seleccionar **2 variables categóricas** (sin incluir el target `renuncia`) y calcular la frecuencia de cada valor.

**1c.** Calcular la frecuencia relativa de la tasa de renuncia.

**1d.** Calcular la tasa de renuncia por `area`. ¿Qué área presenta mayor riesgo?

**Respuesta 1d:**

---

## Punto 2 — Análisis descriptivo gráfico

**2a.** Graficar un boxplot de `salario_mensual` separado por `renuncia`. Interpretar: ¿los empleados que renuncian tienen salarios distintos?

**2b.** Graficar en un histograma la variable 'salario_mensual'

**2c.** Graficar la variable `antiguedad`.

---

## Punto 3 — Modelo
**Ejecutar el siguiente código sin modificarlo.**

In [ ]:
# ── NO MODIFICAR ──
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder

vars_modelo = ['edad', 'salario_mensual', 'antiguedad', 'horas_semanales',
               'satisfaccion', 'cant_proyectos', 'area', 'cargo', 'viaja_trabajo']

df_modelo = df[vars_modelo + ['renuncia']].copy()
for col in ['area', 'cargo', 'viaja_trabajo']:
    df_modelo[col] = LabelEncoder().fit_transform(df_modelo[col])

X = df_modelo[vars_modelo]
y = df_modelo['renuncia']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=NRO_MATRICULA
)

arbol = DecisionTreeClassifier(max_depth=5, random_state=NRO_MATRICULA)

cv_scores = cross_val_score(arbol, X_train, y_train, cv=5, scoring='accuracy')

arbol.fit(X_train, y_train)
preds = arbol.predict(X_test)

print(f"CV scores   : {cv_scores.round(3)}")
print(f"CV media    : {cv_scores.mean():.4f}")
print(f"CV std      : {cv_scores.std():.4f}")
print(f"Predicciones generadas: {len(preds)}")

---

## Punto 4 — Análisis SHAP

In [ ]:
# ── NO MODIFICAR ──
import subprocess, sys
def install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
try:
    import shap
    from packaging.version import Version
    if Version(shap.__version__) < Version('0.46'): raise ImportError
except ImportError:
    install('shap>=0.46'); import shap

explainer   = shap.TreeExplainer(arbol)
shap_values = explainer(X_test)[:, :, 1]
print(f"shap_values shape: {shap_values.values.shape}")

**4a.** Graficar `shap.plots.bar`. ¿Qué variable tiene mayor importancia global?

**4b.** Graficar `shap.plots.beeswarm`. Elegir **una variable** e interpretar qué dice SHAP sobre su efecto en la predicción de renuncia.